# Connect Four, gate gtactic - Qwen3-8B
Plays `grids/duel_c4_tactic.yaml` for **Qwen/Qwen3-8B-FP8** (same modes as the 4B notebook).
The KV cache is kept in bf16 (`NO_FP8_KV_CACHE=1`, see `profiles.py`): the 8B Connect Four games already played (e1, 0 and the memory grid) were played that way on the remote machine, so the new games are played the same way. A Colab L4 has room for it (Connect Four prompts are about 3k tokens).

# Setup
Clones the repository (only the benchmark folder: the two-player games need no game files), installs vLLM, python-chess and Stockfish, compiles the Connect Four solver (`c4solver.py`: the oracle of the gate modes, it judges every move) and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        # the latest code and the latest results pushed to git (they decide what is skipped, with Drive)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0", "pyyaml", "chess"], check=True)
    subprocess.run(["apt-get", "install", "-qq", "-y", "stockfish"], check=True) # the sf<N> / elo<N> opponents
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
import c4solver
if not c4solver.available(): # the Connect Four solver: the oracle of the gate modes, judges every move (g++, make, curl)
    c4solver.setup()
print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Benchmark


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

Wed Nov 11 22:48:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   65C    P8             17W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# What to run
The grid and the models are fixed for this notebook. `limit`: at most this many games per model (`None`: all; e.g. `4` for a first look).
`n_parallel = "auto"`: as many games at once as fit in the GPU memory (see `engine.ContextBudget`).

The first cell below prints, for each results folder, how many games of the grid it already holds: the games found in Drive **or** in the git clone are skipped. Results played on another machine are seen only once they are committed and pushed.

In [4]:
os.environ["NO_FP8_KV_CACHE"] = "1" # the bf16 cache, as the 8B Connect Four games already played (profiles.py)
from records import LEGACY_RESULTS
from run import played_by_folder, summary
from run_duel import load_duel_grid, run_duel_grid

grid = load_duel_grid("grids/duel_c4_tactic.yaml")
models = ["Qwen/Qwen3-8B-FP8"]
limit = None
n_parallel = "auto"

# Where the games already played are looked for: a game found in ANY of these folders is not played again.
#   - Drive: the games played on Colab (new games are saved here)
#   - the git clone (Code/Benchmark/results): the games committed and pushed from anywhere (the remote machine, other
#     Colab runs copied into the repo). The setup cell pulls before this, so it is the latest pushed state.
if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results"
read = [LEGACY_RESULTS] if os.path.abspath(LEGACY_RESULTS) != os.path.abspath(results) else []
print("New games saved in:", results)
print("Also read:", read, "\n")
played_by_folder(grid, [results, *read], models)
print()
summary(grid, results, read, models)

New games saved in: /content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results
Also read: ['/content/LLMs-Play-Textual-Games/Code/Benchmark/results'] 

/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results: 135 games of the grid, 37 only here
/content/LLMs-Play-Textual-Games/Code/Benchmark/results: 212 games of the grid, 114 only here
played in at least one folder: 249 of 250

model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-8B-FP8                            gtactic      50     0            0.0
Qwen/Qwen3-8B-FP8                            lgtactic      50     0            0.0
Qwen/Qwen3-8B-FP8                            grand28      50     0            0.0
Qwen/Qwen3-8B-FP8                                e1      49     1            0.0
Qwen/Qwen3-8B-FP8                                 0      50     0            0.0
total: 249 played, 1 left (the token estimate uses some games already played with the same mode, models, games an

# Run
Every game is saved on Drive as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept; run the notebook again to resume.
If loading the second model fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [5]:
run_duel_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)
summary(grid, results, read, models)

Qwen/Qwen3-8B-FP8: 1 games to play


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:03<00:00, 10.32it/s]


Qwen/Qwen3-8B-FP8: KV cache of 68784 tokens


1 games running, 0 waiting, 0 ended; since the start 0.0 games/hour, 1589 tokens/min; last 10 min 0.0 games/hour, 1706 tokens/min; KV cache: target 92% (games started on the measured usage), actual 5% on average since the start, 6% in the last 10 min (peak 10%); 0 requests paused (preempted) in the last 10 min, 0 since the start

('Qwen/Qwen3-8B-FP8', 'connect_four', 'plain', 'e1', 'self-e1', 50): steps 7, score 0/1, 2336 s
Qwen/Qwen3-8B-FP8: done in 39 min
model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-8B-FP8                            gtactic      50     0            0.0
Qwen/Qwen3-8B-FP8                            lgtactic      50     0            0.0
Qwen/Qwen3-8B-FP8                            grand28      50     0            0.0
Qwen/Qwen3-8B-FP8                                e1      50     0            0.0
Qwen/Qwen3-8B-FP8                                 0      50     0            0.0
total: 250 played, 0 left (the token estimate uses some games already played with the same mode, models, games and prompts of the grid)


# Quick look
Per model and mode, from Drive and git together: games, score against self-e1, thinking rate, thinking tokens per game, and the one-move tactics (`gate_analysis.tactics_summary`): wins in one taken, single threats blocked.

In [ ]:
import gate_analysis as ga
from records import load_table

folders = [results, *read]
table = load_table(*folders)
games = table[table.game.isin(grid["games"]) & table.model.isin(models or grid["models"]) & table["mode"].isin(grid["modes"])
              & table.prompt.isin(grid["prompts"]) & table.seed.isin(grid["seeds"]) & table.level.isin(grid["games"]["connect_four"])]
display(games.groupby(["model", "mode"]).agg(games=("seed", "size"), score=("norm_score", "mean"),
        think_rate=("think_rate", "mean"), kthink=("tokens_generated_thinking", lambda t: t.mean() / 1000)).round(2))
moves = ga.move_table(*folders, rebuild=False)
moves = moves[moves.model.isin(models or grid["models"]) & moves["mode"].isin(grid["modes"]) & moves.prompt.isin(grid["prompts"])
              & moves.seed.isin(grid["seeds"]) & moves.level.isin(grid["games"]["connect_four"])
              & (moves.player_mode == moves["mode"])] # the moves of the mode, not of its e1 opponent
ga.tactics_summary(moves, by=("model", "player_mode"))[["moves", "think_pct", "took_win", "took_win_pct", "blocked", "blocked_pct"]]